In [20]:
import ollama

language_model = "hf.co/unsloth/Qwen3-0.6B-GGUF:UD-Q4_K_XL"
embedding_model = "hf.co/CompendiumLabs/bge-base-en-v1.5-gguf:Q4_K_M"

In [6]:
datasets=[]

with open('datasets/cat-facts.txt','r',encoding='utf-8') as f:
    datasets=f.readlines()

    print(f"Loaded {len(datasets)} cat facts")

Loaded 150 cat facts


In [17]:
VECTOR_DB=[]

In [18]:
def add_chunk_to_vector_db(chunk):
    embedding=ollama.embed(model=embedding_model,input=chunk)['embeddings'][0]
    VECTOR_DB.append((chunk,embedding))

In [ ]:
for i , chunk in enumerate(datasets):
    add_chunk_to_vector_db(chunk)
    if i%10==0:
        print(f"Added chunk {i+1}/{len(datasets)} to vector database")

In [24]:
def cosine_similarity(vec1,vec2):
    dot_product=sum(a*b for a,b in zip(vec1,vec2))
    
    magitude1=sum(a**2 for a in vec1)**0.5
    magitude2=sum(b**2 for b in vec2)**0.5

    if magitude1==0 or magitude2==0:
        return 0
    return dot_product/(magitude1*magitude2)

In [25]:
def retriever(query, top_k=3):
    query_embedding = ollama.embed(model=embedding_model, input=query)['embeddings'][0]
    similarities = []
    for chunk, embedding in VECTOR_DB:
        sim = cosine_similarity(query_embedding, embedding)
        similarities.append((chunk, sim))
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]

In [ ]:
query = "What is the average lifespan of a cat?"


In [26]:
result=retriever(query)
print("Top relevant chunks:")
for i, chunk in enumerate(result):
    print(f"{i+1}, {chunk}")

Top relevant chunks:
1, ('When well treated, a cat can live twenty or more years but the average life span of a domestic cat is 14 years.\n', 0.8955949610774252)
2, ('The oldest cat on record was Crème Puff from Austin, Texas, who lived from 1967 to August 6, 2005, three days after her 38th birthday. A cat typically can live up to 20 years, which is equivalent to about 96 human years.\n', 0.7758037392291062)
3, ('On average, cats spend 2/3 of every day sleeping. That means a nine-year-old cat has been awake for only three years of its life.\n', 0.7616981201610812)


In [31]:
instruction = "You are a helpful assistant that answers questions based on the provided context. Use the retrieved chunks to formulate your answer. If the information is not sufficient, say you don't know."

In [32]:
stream=ollama.chat(
    model=language_model,
    messages=[
        {'role':'system','content':instruction},
        {'role':'user','content':query}
    ],

    stream=True
)

In [33]:
print("Chatbot response: ")

for chunk in stream:
    print(chunk['message']['content'],end='',flush=True)

Chatbot response: 
<think>
Okay, I need to find out the average lifespan of a cat. Let me think. I remember that cats are generally longer than dogs, but I'm not exactly sure. I think some sources say they can live up to 12 to 15 years. Wait, but I should check if that's accurate.

Wait, I think I've heard that cats can live up to 15 years, but some might be shorter. Maybe the average is around 8 to 12 years. Let me make sure. Oh, right, in some countries, like Japan, they might have longer lifespans. Also, factors like genetics and environment play a role. But the average overall is 8 to 12 years. I need to confirm this with the information given. Since the user provided the context, I should use that. The answer should be in the box and be in English. Alright, I think that's it.
</think>

The average lifespan of a cat is typically around 8 to 12 years, depending on factors like genetics, care, and environment.